In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("hehehe").getOrCreate()

spark.sql("""
    SELECT brand_name, count(*) as so_luong_sp
    FROM local_catalog.tiki.products VERSION AS OF 1933013413088474368
    GROUP BY brand_name 
    ORDER BY so_luong_sp DESC
    LIMIT 5
""").show(truncate=False)


+---------------------------+-----------+
|brand_name                 |so_luong_sp|
+---------------------------+-----------+
|La Roche-Posay             |81         |
|The Cocoon Original Vietnam|75         |
|dermalogica                |48         |
|Hada Labo                  |43         |
|Dabo                       |40         |
+---------------------------+-----------+



In [6]:
# Lệnh Rollback khôi phục vĩnh viễn
spark.sql("CALL local_catalog.system.rollback_to_snapshot('tiki.products', 1933013413088474368)")

print("Đã thực hiện Load Game thành công! Dữ liệu đã được cứu.")

# Kiểm tra lại bảng hiện tại xem dữ liệu đã thực sự về chưa
spark.sql("""
    SELECT brand_name, count(*) as so_luong_sp
    FROM local_catalog.tiki.products 
    GROUP BY brand_name 
    ORDER BY so_luong_sp DESC
    LIMIT 5
""").show(truncate=False)

Đã thực hiện Load Game thành công! Dữ liệu đã được cứu.
+---------------------------+-----------+
|brand_name                 |so_luong_sp|
+---------------------------+-----------+
|La Roche-Posay             |81         |
|The Cocoon Original Vietnam|75         |
|dermalogica                |48         |
|Hada Labo                  |43         |
|Dabo                       |40         |
+---------------------------+-----------+



In [7]:
print("1. Xem lại số lượng điểm lưu hiện tại:")
spark.sql("SELECT * FROM local_catalog.tiki.products.history").show(truncate=False)

print("2. Kích hoạt lệnh dọn rác (Chỉ giữ lại 1 bản lưu mới nhất):")
# Gọi thủ tục hệ thống của Iceberg để xóa các snapshot cũ khỏi ổ cứng MinIO
spark.sql("CALL local_catalog.system.expire_snapshots(table => 'tiki.products', retain_last => 1)").show()
print("Đã dọn dẹp sạch sẽ ổ cứng!")

print("3. Xem lại lịch sử sau khi dọn:")
spark.sql("SELECT * FROM local_catalog.tiki.products.history").show(truncate=False)

1. Xem lại số lượng điểm lưu hiện tại:
+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-12 09:28:05.391|562330036124825977 |NULL               |false              |
|2026-05-12 09:30:07.073|1346218287291490800|NULL               |false              |
|2026-05-12 09:31:40.491|6907064054326218913|NULL               |false              |
|2026-05-12 09:33:29.226|1933013413088474368|NULL               |true               |
|2026-05-12 15:34:06.952|4311570158704368180|1933013413088474368|false              |
|2026-05-12 15:34:44.952|5073778695285372701|4311570158704368180|false              |
|2026-05-12 15:54:29.146|1933013413088474368|NULL               |true               |
+-----------------------+-------------------+-------------------+-------------------+

2. Kích hoạt l